完整的 MinGPT 函数

流程：

x = TokenEmbedding_batch(X) 

x = LearnablePositionalEncoding(x)

x = TransformerBlock(x)（先堆 1 个 Block，跑通再加）

x = FinalLayerNorm(x)

logits = LinearHead(x)

输入：X [B, T] → 输出：logits [B, T, V]

其中：

TransformerBlock 函数严格按 Pre-LN 结构：

residual1 = x + SingleHeadCausalSelfAttention_batch(LayerNorm(x))

residual2 = residual1 + FeedForward(LayerNorm(residual1))

输入 / 输出：[B, T, D]

In [188]:
import numpy as np
import torch
import math
import torch.nn.functional as F

T_max=128 #预设的最大序列长度
D=4 #隐藏维度
V=26 #词表大小
char_to_idx={} #字符到整数索引的映射,形状：{V}（V 是字符表大小）
idx_to_char={} #整数索引回字符的映射,形状：{V}


for i in range(26):
    char_to_idx[chr(ord('a')+i)]=i
    idx_to_char[i]=chr(ord('a')+i)

In [189]:
embedding_matrix = torch.rand(26,D) #向量化，先随便填的

def TokenEmbedding(token_ids): #输入：token_ids [T] → 输出：x [T, D]（D 是隐藏维度）。或者输入：token_ids [B, T] → 输出：x [B, T, D]。高级索引机制，就是好用。
    return torch.tensor(embedding_matrix[token_ids]) #向量化

In [190]:
pos_embedding_matrix=torch.rand(T_max, D) #位置编码变量,形状[T_max, D]

def LearnablePositionalEncoding(X): #先不用 RoPE，用简单可学习位置编码；输入：x [T, D] → 输出：x + pos_embedding_matrix [T, D]
    
    return X+pos_embedding_matrix[:len(X),:].unsqueeze(0) #.unsqueeze(0) 是 PyTorch 中用于增加张量维度的函数，具体作用是在第 0 维（即最前面）插入一个大小

In [191]:
Wq=torch.rand(D,D) #生成 Q/K/V 的三个权重矩阵，每个形状：[D, D]。没训练前先随机
Wk=torch.rand(D,D)
Wv=torch.rand(D,D)
def SingleHeadCausalSelfAttention(X): #输入：x [B, T, D] → 输出：attn_output [T, D]
    Q=torch.matmul(X,Wq) #直接乘是没问题的，pytorch支持。
    K=torch.matmul(X,Wk)
    V=torch.matmul(X,Wv)
    

    S=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(D) #原始注意力分数[T, T]
    
    T=S.shape[0]
    M=torch.zeros((T,T),dtype=torch.float32)
    rows,cols=torch.triu_indices(T,T,offset=1) #矩阵的上三角，offset=1表示主对角右移1
    M[rows,cols]=-torch.inf #将上三角位置设为负无穷
    S=S+M #注意力掩码处理后的分数[T, T]

    exp_S=torch.exp(S)
    A=exp_S/torch.sum(exp_S,axis=1,keepdims=True) #注意力权重

    O=torch.matmul(A,V) #计算投影

    '''
    print(Q)
    print(K)
    print(V)
    print(S)
    print(M)
    print(A)
    '''
    return O

In [192]:
def LayerNorm(X): #层归一化
    
    mu=torch.mean(X,dim=-1,keepdims=True)
    sigma=torch.var(X,axis=-1,keepdims=True)
    normalized_X=(X-mu)/torch.sqrt(sigma+1e-5)
    return normalized_X

In [193]:
W1=torch.rand(D,4*D)
b1=torch.rand(4*D)
W2=torch.rand(4*D,D)
b2=torch.rand(D)
def FeedForward(X):
    X=torch.matmul(X,W1)+b1
    X=F.gelu(X)
    X=torch.matmul(X,W2)+b2
    return X

In [ ]:
def TransformerBlock(X): #输入 / 输出：[B, T, D]。问：归一化之后token和token之间的相对大小不就变了吗，还怎么区分token和token？答：归一化之后token向量方向不变，长度变了。长度是一维信息，在方向里多加一维就效果一样了。
    X = X + SingleHeadCausalSelfAttention(LayerNorm(X))
    X = X + FeedForward(LayerNorm(X))
    return X

In [195]:
def FinalLayerNorm(X): #就是普通的 LayerNorm
    return LayerNorm(X)

In [196]:
W_head=torch.rand(D,V)
b_head=torch.rand(V)
def LinearHead(X): #线性层，输入 [B, T, D] → 输出 [B, T, V]，参数 W_head [D, V], b_head [V]。
    X=torch.matmul(X,W_head)+b_head
    return X

In [197]:
def miniGPT(token_ids):
    X = TokenEmbedding(token_ids) 
    
    X = LearnablePositionalEncoding(X)

    X = TransformerBlock(X)

    X = FinalLayerNorm(X)

    logits = LinearHead(X)

    return logits

In [ ]:
if __name__ == "__main__":
    input_str=[['h','e','l','l','o']] #输入序列，形状：[B, T]（B 是批次大小，T 是序列长度）
    input_ids=torch.tensor([[char_to_idx[c] for c in seq] for seq in input_str])
    logits=miniGPT(input_ids)
    print(f"输入形状: {input_ids.shape}")
    print(f"输出logits形状: {logits.shape}")

/tmp/ipykernel_154559/2575102866.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(embedding_matrix[token_ids]) #向量化


<class 'torch.Tensor'>
tensor([[[-0.2574,  1.0877,  0.5714,  0.2573,  0.1433,  1.0199,  0.2530,
           1.8502,  0.9989,  0.4049,  1.8653, -0.5275,  0.1143,  0.3445,
           1.1067, -0.0442,  0.8381,  1.6193, -0.9178,  1.4410,  0.7454,
           1.0410,  1.4935,  0.2041, -0.3968,  0.8406],
         [-0.6779,  1.5757,  0.6415,  0.0154,  0.5554,  0.8863,  0.6765,
           1.6323,  0.6611,  0.2217,  1.9204, -0.2917,  0.5233,  1.0835,
           1.3913, -0.1844,  0.2465,  1.1321, -0.6789,  1.4424,  1.1968,
           1.1024,  1.7091,  0.2227,  0.2436,  0.9853],
         [-0.3050,  1.1108,  0.4416,  0.1061,  0.3718,  1.1934, -0.2596,
           1.9435,  1.5751,  0.2513,  1.5761, -0.5323, -0.0650,  0.0855,
           1.2855,  0.3981,  0.9222,  1.4503, -0.8325,  1.2077,  1.0197,
           1.2819,  1.2767,  0.3479,  0.0441,  0.5412],
         [-0.3050,  1.1108,  0.4416,  0.1061,  0.3718,  1.1934, -0.2596,
           1.9435,  1.5751,  0.2513,  1.5761, -0.5323, -0.0650,  0.0855,
      